In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_6.pickle

In [ ]:
%%cudf.pandas.profile
### cell 6 ###

# Narrow down columns to reduce GPU data movement
li = lineitem[['L_PARTKEY', 'L_SUPPKEY', 'L_ORDERKEY',
               'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_QUANTITY']]
fpart = part[['P_PARTKEY']][part.P_NAME.str.contains('ghost')]
sp = supplier[['S_SUPPKEY', 'S_NATIONKEY']]
na = nation[['N_NATIONKEY', 'N_NAME']]
ps = partsupp[['PS_PARTKEY', 'PS_SUPPKEY', 'PS_SUPPLYCOST']]
ord = orders[['O_ORDERKEY', 'O_ORDERDATE']]

# Chain merges, compute TMP and O_YEAR in one assign
df = (li.merge(fpart, left_on='L_PARTKEY', right_on='P_PARTKEY')
        .merge(sp, left_on='L_SUPPKEY', right_on='S_SUPPKEY')
        .merge(na, left_on='S_NATIONKEY', right_on='N_NATIONKEY')
        .merge(ps, left_on=['L_PARTKEY','L_SUPPKEY'], right_on=['PS_PARTKEY','PS_SUPPKEY'])
        .merge(ord, left_on='L_ORDERKEY', right_on='O_ORDERKEY')
        .assign(
            TMP=lambda x: x.L_EXTENDEDPRICE * (1 - x.L_DISCOUNT) - (x.PS_SUPPLYCOST * x.L_QUANTITY),
            O_YEAR=lambda x: x.O_ORDERDATE.dt.year
        ))

# Group and sort
total = (df.groupby(['N_NAME','O_YEAR'], as_index=False, sort=False)['TMP']
           .sum()
           .sort_values(['N_NAME','O_YEAR'], ascending=[True, False]))